# MLP

Plain PyTorch MLP on `X_train_log_knn.csv`, evaluated on the shared folds (`Stacking/folds.csv`).

**Final setup:** 1 hidden layer of 64 neurons, ReLU, AdamW (`lr = 1e-3`, `weight_decay = 1`), MSE loss, 200 epochs, batches of 64, X and y standardized inside each fold. The prediction is the average of 5 networks trained with seeds 0–4.

**Result:** out-of-fold R² **0.686** (SVR on the same data: 0.708). In the stack it gets a weight of about 0.15 and adds +0.001.

### What was tried (fold 0 unless stated)

| Change | Val R² | Kept? |
|---|---|---|
| Start: 2 × 128, Tanh, Adam, no regularization | 0.505 (train 0.99) | |
| + weight decay 1 (tried 0 – 10) | 0.663 | ✅ |
| + dropout 0.1 – 0.5 | 0.62 – 0.66 | ❌ no gain |
| + average of 5 seeds (single seeds: 0.62 – 0.65) | 0.665 | ✅ |
| 1 hidden layer instead of 2 (Tanh, 32 neurons) | 0.710 | ✅ |
| Deeper or funnel networks (2–3 small layers) | 0.68 – 0.70 | ❌ |
| ReLU, 1 × 64 | 0.715 | ✅ |
| ELU | 0.68 – 0.69 | ❌ |
| 5 folds: `knn` → `log_knn` | 0.665 → **0.686** | ✅ |
| 5 folds: Huber loss, delta 0.25 – 1 | 0.684 – 0.689 | ❌ no gain |

In [1]:
## Imports

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

In [2]:
## Data loading

X = pd.read_csv("X_train_log_knn.csv", index_col="id")
y = pd.read_csv("y_train.csv", index_col="id").loc[X.index, "y"]           # keep the same 1152 rows as X, in the same order
fold = pd.read_csv("Stacking/folds.csv", index_col="id").loc[X.index, "fold"]
X_test = pd.read_csv("X_test_log_knn.csv", index_col="id")

print(f"X {X.shape} | y {y.shape} | X_test {X_test.shape}")

X (1152, 216) | y (1152,) | X_test (776, 216)


In [3]:
## Settings

N_HIDDEN_LAYERS = 1
NEURONS = 64
WEIGHT_DECAY = 1
LEARNING_RATE = 1e-3
NUM_EPOCHS = 200
BATCH_SIZE = 64
N_SEEDS = 5

In [4]:
## NeuralNet class

class NeuralNet(nn.Module):

    def __init__(self, input_dimension, output_dimension, n_hidden_layers, neurons, retrain_seed):
        super(NeuralNet, self).__init__()
        self.activation = nn.ReLU()
        self.input_layer = nn.Linear(input_dimension, neurons)
        self.hidden_layers = nn.ModuleList([nn.Linear(neurons, neurons) for _ in range(n_hidden_layers - 1)])
        self.output_layer = nn.Linear(neurons, output_dimension)
        self.retrain_seed = retrain_seed
        self.init_xavier()

    def forward(self, x):
        x = self.activation(self.input_layer(x))
        for l in self.hidden_layers:
            x = self.activation(l(x))
        return self.output_layer(x)

    def init_xavier(self):
        # Random seed for weight initialization
        torch.manual_seed(self.retrain_seed)

        def init_weights(m):
            if type(m) == nn.Linear:
                torch.nn.init.xavier_uniform_(m.weight, gain=nn.init.calculate_gain('relu'))
                m.bias.data.fill_(0)

        self.apply(init_weights)

In [5]:
## Fit function

def fit(model, training_set, num_epochs, optimizer):
    for epoch in range(num_epochs):
        for x_batch, y_batch in training_set:
            optimizer.zero_grad()
            loss = torch.mean((model(x_batch).reshape(-1, ) - y_batch.reshape(-1, )) ** 2)   # MSE
            loss.backward()
            optimizer.step()

In [6]:
## Train on some rows, predict others (average of N_SEEDS networks)

def train_predict(X_train, y_train, X_pred):
    # scaling fitted on the training rows only
    x_scaler = StandardScaler().fit(X_train)
    y_scaler = StandardScaler().fit(y_train.to_frame())
    X_tr = torch.tensor(x_scaler.transform(X_train), dtype=torch.float32)
    y_tr = torch.tensor(y_scaler.transform(y_train.to_frame()), dtype=torch.float32)
    X_pr = torch.tensor(x_scaler.transform(X_pred), dtype=torch.float32)
    training_set = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)

    preds = []
    for seed in range(N_SEEDS):
        model = NeuralNet(input_dimension=X_tr.shape[1], output_dimension=1, n_hidden_layers=N_HIDDEN_LAYERS,
                          neurons=NEURONS, retrain_seed=seed)
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        torch.manual_seed(seed)                                   # also fixes the order of the shuffled batches
        fit(model, training_set, NUM_EPOCHS, optimizer)
        model.eval()
        with torch.no_grad():
            preds.append(y_scaler.inverse_transform(model(X_pr).numpy()).ravel())   # back to years
    return np.mean(preds, axis=0)

In [7]:
## 5-fold cross-validation (shared folds) -> Stacking/oof_mlp.csv

oof = pd.Series(np.nan, index=X.index, name="mlp")
for f in range(5):
    train, val = fold != f, fold == f
    oof[val] = train_predict(X[train], y[train], X[val])
    print(f"fold {f}: R2 {r2_score(y[val], oof[val]):.4f}")

print(f"out-of-fold R2: {r2_score(y, oof):.4f}")
oof.to_csv("Stacking/oof_mlp.csv")

fold 0: R2 0.7148
fold 1: R2 0.6830
fold 2: R2 0.6334
fold 3: R2 0.6895
fold 4: R2 0.7029
out-of-fold R2: 0.6861


In [8]:
## Final model: train on all 1152 rows, predict the 776 test rows -> MLP/submission_mlp.csv

submission = pd.Series(train_predict(X, y, X_test), index=X_test.index, name="y")
submission.to_csv("MLP/submission_mlp.csv")
print(submission.describe().round(1))

count    776.0
mean      69.8
std        8.3
min       48.1
25%       64.2
50%       70.4
75%       75.9
max       87.6
Name: y, dtype: float64
